In [1]:
import numpy as np
import pandas as pd
import glob
import os
import h5py
from mne.filter import filter_data, notch_filter
import gc as garbageCollector
from scipy.signal import fftconvolve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# %matplotlib qt
from mne.time_frequency import psd_array_multitaper
import scipy.io as sio
from scipy.signal import detrend
from tqdm import tqdm
from collections import Counter
from bisect import bisect

In [2]:
from datetime import datetime, time as datetime_time, timedelta

def time_diff(start, end):
    if isinstance(start, datetime_time): # convert to datetime
        assert isinstance(end, datetime_time)
        start, end = [datetime.combine(datetime.min, t) for t in [start, end]]
    if start <= end: # e.g., 10:33:26-11:15:49
        return end - start
    else: # end < start e.g., 23:55:00-00:25:00
        end += timedelta(1) # +day
        assert end > start
        return end - start
    
    
def get_grass_start_end_time(starttime_raw, endtime_raw):
    
    time_str_elements = starttime_raw.flatten()
    start_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))
    time_str_elements = endtime_raw.flatten()
    end_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))

    start_time = start_time.split(':')
    second_elements = start_time[-1].split('.')
    start_time = datetime.datetime(1990,1,1,hour=int(float(start_time[0])), minute=int(float(start_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))
    end_time = end_time.split(':')
    second_elements = end_time[-1].split('.')
    end_time = datetime.datetime(1990,1,1,hour=int(float(end_time[0])), minute=int(float(end_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))

    return start_time, end_time


def check_load_mgh_dataset_with_label(signal_path, label_path, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signal_path)
        data_path = os.path.basename(signal_path)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        if 'grass' in signal_path:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
        if 'grass' in signal_path:
            with h5py.File(label_path) as ffl:
                sleep_stage = ffl['stage'][()].flatten()
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signal_path,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [ ''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0]) ]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signal_path:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

        # load labels
        with h5py.File(label_path) as ffl:
            sleep_stage = ffl['stage'][()].flatten()
            if 'grass' in signal_path:
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
            ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
    if sleep_stage.shape[0]!=signal.shape[1]:
        raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        signal_channel_names = channel_names
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        signal_channel_names = []
        for ichannel in ['ABD', 'CHEST']:
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        signal_channel_names = []
        for i in range(len(channels)):
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]#.T

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

    # check whether sleep_stage contains all 5 stages
    stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
    if len(stages)<=1:
        raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'channel_ids': signal_channel_ids, 'channel_names': signal_channel_names}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, sleep_stage, params

In [3]:
import os
path_mad = '/media/mad3/Datasets_ConvertedData/'
path_grass = path_mad + 'sleeplab/grass_data/'
files_grass =[]
files_grass = [f for f in sorted(os.listdir(path_grass))]
files_grass = files_grass[1:]

# path_selected_subjects = '/home/snasiri/Dropbox (Partners HealthCare)/EMGs/figures'
files =[]
files = [f for f in sorted(os.listdir(path_selected_subjects))]

for k in range(2,3): # len(files)): # subject ID
    
    import datetime
    file_subject = files [k]
    print ("subject = ", file_subject)

    Files = os.listdir(path_grass + file_subject)
    LabelFile = [match for match in Files if "annotations" in match]
    SignalFile = [match for match in Files if "Signal" in match]
    LabelFile_mat = [match for match in Files if "Labels" in match]

    
    if LabelFile and SignalFile and LabelFile_mat: 
         
        
        signal_path = path_grass + file_subject + '/' + SignalFile [0]
        label_path = path_grass + file_subject + '/' + LabelFile [0]
        label_mat_path = path_grass + file_subject + '/' + LabelFile_mat [0]

        signal, sleep_stage, params = check_load_mgh_dataset_with_label(signal_path, label_mat_path)
        fs = params ['Fs']
        annotation = pd.read_csv(label_path)

        test = annotation.copy()
        start_time = annotation.iloc[0]['time']
        test = test[test['duration'].notna()]
        columnsName = test.event.unique().tolist()
        from datetime import datetime
        FMT = '%H:%M:%S'
        s1 = test.iloc[-1]['time']
        s2 = test.iloc[0]['time']
        tdelta = datetime.strptime(s1, FMT) - datetime.strptime(s2, FMT)
        len_in_seconds=tdelta.seconds
        label = pd.DataFrame(index=np.arange(signal.shape[1]) , columns=columnsName)
        start_time = test.iloc [0]['time']
        start_time = datetime.strptime(start_time, '%H:%M:%S')
        for i in range(len(test)):
            current_time = test.iloc[i]['time']
            current_time = datetime.strptime(current_time, '%H:%M:%S')
            tdelta = time_diff(start_time , current_time) 
            len_in_seconds=tdelta.seconds
            time_samples_index = int (len_in_seconds * fs)
            duration_of_event = int (test.iloc [i]['duration'] * fs)
            if duration_of_event < 1:
                duration_of_event = int(np.ceil(duration_of_event))   
            event_name = test.iloc [i]['event']
            label.at [time_samples_index:(time_samples_index + duration_of_event -1), event_name] = 1


    with h5py.File(label_mat_path) as ffl:
        apnea = ffl['apnea'][()].flatten()  

    mylist = label.columns.tolist()
    re = 'Respiratory Event'
    respiratory_event = [s for s in mylist if re.lower() in s.lower()] 

    respiratory_event_interest = label[respiratory_event]
    respiratory_event_interest = respiratory_event_interest.fillna(0)

    
    Hypopnea = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Hypopnea' in s]
    obstructive_apnea = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Obstructive Apnea' in s]
    central_apnea = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Central Apnea' in s]
    mixed_apnea = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Mixed Apnea' in s]
    arousal_respiration = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Arousal' in s]
    hypovent = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'Hypoventilation' in s]
    RERA = [s for i, s in enumerate(respiratory_event_interest.columns.tolist()) if 'RERA' in s]

    columns_resp = ['no event', 'obstructive apnea', 'central apnea', 'mixed apnea', 'hypopnea', 'hypoventilation', 'RERA']
    respiratory_event_label = pd.DataFrame(index=np.arange(0,len(respiratory_event_interest)), columns=columns_resp)
    respiratory_event_label = respiratory_event_label.fillna(0) # with 0s rather than NaNs

    if Hypopnea:
        respiratory_event_label ['hypopnea'] =  respiratory_event_interest [Hypopnea[0]]
        if len(Hypopnea)>1:
            for i in Hypopnea[1:]:
                respiratory_event_label ['hypopnea'] = respiratory_event_label ['hypopnea']  + respiratory_event_interest [i]

    if obstructive_apnea:
        respiratory_event_label ['obstructive apnea'] =  respiratory_event_interest [obstructive_apnea[0]]
        if len(obstructive_apnea)>1:
            for i in obstructive_apnea[1:]:
                respiratory_event_label ['obstructive apnea'] = respiratory_event_label ['obstructive apnea']  + respiratory_event_interest [i]

    if central_apnea:
        respiratory_event_label ['central apnea'] =  respiratory_event_interest [central_apnea[0]]
        if len(central_apnea)>1:
            for i in central_apnea[1:]:
                respiratory_event_label ['central apnea'] = respiratory_event_label ['central apnea']  + respiratory_event_interest [i]

    if mixed_apnea:

        respiratory_event_label ['mixed apnea'] =  respiratory_event_interest [mixed_apnea[0]]

        if len(mixed_apnea)>1:
            for i in mixed_apnea[1:]:
                respiratory_event_label ['mixed apnea'] = respiratory_event_label ['mixed apnea']  + respiratory_event_interest [i]


    if hypovent:
        respiratory_event_label ['hypoventilation'] =  respiratory_event_interest [hypovent[0]]
        if len(hypovent)>1:
            for i in hypovent[1:]:
                respiratory_event_label ['hypoventilation'] = respiratory_event_label ['hypoventilation']  + respiratory_event_interest [i]

    if RERA:
        respiratory_event_label ['RERA'] =  respiratory_event_interest [RERA[0]]
        if len(RERA)>1:
            for i in RERA[1:]:
                respiratory_event_label ['RERA'] = respiratory_event_label ['RERA']  + respiratory_event_interest[i]

                
     
    PLM_event_interest = label[PLM_events]
    PLM_event_interest = PLM_event_interest.fillna(0)
    
    ['PLM - Isolated', '* Arousal - PLM', 'PLM - Periodic']
    
    Periodic_PLM = [s for i, s in enumerate(PLM_event_interest.columns.tolist()) if 'Periodic' in s]
    Arousal_PLM = [s for i, s in enumerate(PLM_event_interest.columns.tolist()) if 'Arousal' in s]
    Isolated_PLM = [s for i, s in enumerate(PLM_event_interest.columns.tolist()) if 'Isolated' in s]
    

    columns_PLM = ['no event', 'PLM - Isolated', 'Arousal - PLM', 'PLM - Periodic']
    PLM_event_label = pd.DataFrame(index=np.arange(0,len(PLM_event_interest)), columns=columns_PLM)
    PLM_event_label = PLM_event_label.fillna(0) # with 0s rather than NaNs

    if Periodic_PLM:
        PLM_event_label ['PLM - Periodic'] =  PLM_event_interest [Periodic_PLM[0]]
        if len(Periodic_PLM)>1:
            for i in Periodic_PLM[1:]:
                PLM_event_label ['PLM - Periodic'] = PLM_event_label ['PLM - Periodic']  + PLM_event_interest [i]

    if Arousal_PLM:
        PLM_event_label ['Arousal - PLM'] =  PLM_event_interest [Arousal_PLM[0]]
        if len(Arousal_PLM)>1:
            for i in Arousal_PLM[1:]:
                PLM_event_label ['Arousal - PLM'] = PLM_event_label ['Arousal - PLM']  + PLM_event_interest [i]

    if Isolated_PLM:
        PLM_event_label ['PLM - Isolated'] =  PLM_event_interest [Isolated_PLM[0]]
        if len(Isolated_PLM)>1:
            for i in Isolated_PLM[1:]:
                PLM_event_label ['PLM - Isolated'] = PLM_event_label ['PLM - Isolated']  + PLM_event_interest [i]

    
    #### For annotation file: 0, 1, 2, 3, 4, 5 => ?, W, N1, N2, N3, R
    # 
    columns_sleep = ['Sleep_stage_?','Sleep_stage_W','Sleep_stage_N1','Sleep_stage_N2','Sleep_stage_N3', 'Sleep_stage_REM']
    sleep_interest = pd.DataFrame(index=np.arange(0,len(label)), columns=columns_sleep)
    
    se = 'Sleep_stage'
    sleep_event = [s for s in mylist if se.lower() in s.lower()] 
        
    for stage in sleep_event:
        sleep_interest[stage] = label[stage]
        
    sleep_interest = sleep_interest.fillna(0)

    sleep_label_window = sleep_interest.to_numpy()
    stages = sleep_interest.columns.tolist()
    sum_sleep = np.sum(sleep_label_window, axis = 1)

    idx_without_sleep_label = np.where(sum_sleep == 0) [0]
    the_last_value = np.where(sleep_label_window[idx_without_sleep_label[0]-1] == 1)[0][0]
    sleep_label_window[idx_without_sleep_label,the_last_value] = 1
    annoation_label_encoded = np.where(sleep_label_window == 1)[1]
    
    epoch_time = 30   # [min]
    epoch_step_time = 30    # [min]
    epoch_size = int(round(epoch_time*fs))
    epoch_step = int(round(epoch_step_time*fs))
    start_ids = np.arange(0, signal.shape[1]-epoch_size+1, epoch_step)
    seg_ids = list(map(lambda x:np.arange(x,x+epoch_size), start_ids))
    
    annoation_label_segs = annoation_label_encoded[seg_ids].transpose(1,0)
    annotation_y_30sec = np.zeros(annoation_label_segs.shape[1])

    # segment the sleep stages into 30 sec window
    for i in range(annoation_label_segs.shape[1]):
        values, counts = np.unique(annoation_label_segs[:,i], return_counts=True)
        ind = np.argmax(counts)
        value = values[ind]
        annotation_y_30sec [i] = value
    
    annotation_y_30sec_copy = annotation_y_30sec.copy()
    annotation_y_30sec_copy [annotation_y_30sec==1] = 5
    annotation_y_30sec_copy [annotation_y_30sec==2] = 3
    annotation_y_30sec_copy [annotation_y_30sec==3] = 2
    annotation_y_30sec_copy [annotation_y_30sec==4] = 1
    annotation_y_30sec_copy [annotation_y_30sec==5] = 4
    
    annotation_stages = annotation_y_30sec_copy


FileNotFoundError: [Errno 2] No such file or directory: '/run/user/1001/gvfs/smb-share:server=172.18.60.27,share=mgh-neuro-cdac/Datasets_ConvertedData/sleeplab/grass_data/'

In [13]:
respiratory_event_label

,no event,obstructive apnea,central apnea,mixed apnea,hypopnea,hypoventilation,RERA
0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...
5400595,0,0,0,0,0,0,0
5400596,0,0,0,0,0,0,0
5400597,0,0,0,0,0,0,0
5400598,0,0,0,0,0,0,0
